
# EVT Threshold Diagnostics (from existing CSVs)

This notebook:
- Loads your **block maxima** CSV and **GEV fit results** CSV (with columns you provided).
- Lets you set a **threshold grid** for POT diagnostics (`u ∈ {0.4, 0.5, 0.6, 0.7, 0.8, 0.9}` by default).
- Supports **warm‑up trimming** (drop earliest fraction) and **row caps**.
- Produces:
  - **Mean Residual Life (MRL)** plot
  - **Shape (ξ) stability** vs. threshold (with optional GEV overlay from your CSV)
  - Per‑threshold **GPD QQ** plots
  - **KS p‑values** and **Anderson–Darling** stats summary
- Saves all outputs into a timestamped folder under `/mnt/data/`.


In [ ]:

# ==== Parameters ====
BLOCK_MAXIMA_CSV = "/mnt/data/block_maxima.csv"     # path to your block maxima CSV
GEV_RESULTS_CSV  = "/mnt/data/gev_fit_results.csv"  # path to your GEV results CSV (columns listed below)

# Column detection:
# We'll try these in order if COL_NAME is None. Otherwise we use COL_NAME exactly.
COL_NAME = None
SEC_COLS = ['block_max', 'e2e_sec', 'latency_sec', 'e2e_s', 'latency_s', 'value']
MS_COLS  = ['e2e_ms', 'latency_ms']

# Thresholds (seconds) for POT:
THRESHOLDS_S = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Data handling
MAX_ROWS     = None   # e.g., 1_000_000 to cap rows
WARMUP_FRAC  = 0.0    # e.g., 0.02 for 2% earliest rows
KEEP_WARMUP  = True   # if True, keeps warm-up even if WARMUP_FRAC>0

# Output folder (auto if None)
OUT_DIR = None


In [ ]:

# ==== Imports ====
import os, math, json
from datetime import datetime
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import genpareto, kstest

# Jupyter display options
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


In [ ]:

# ==== Load CSVs ====

def load_from_csv(
    path: str,
    col_name: Optional[str] = None,
    max_rows: Optional[int] = None,
    warmup_frac: float = 0.0,
    keep_warmup: bool = True,
    sec_cols=None,
    ms_cols=None
) -> np.ndarray:
    if sec_cols is None:
        sec_cols = ['block_max', 'e2e_sec', 'latency_sec', 'e2e_s', 'latency_s', 'value']
    if ms_cols is None:
        ms_cols = ['e2e_ms', 'latency_ms']

    df = pd.read_csv(path)
    if max_rows is not None:
        df = df.head(max_rows)
    arr = None
    if col_name and col_name in df.columns:
        arr = df[col_name].to_numpy(dtype=float)
    else:
        for c in sec_cols:
            if c in df.columns:
                arr = df[c].to_numpy(dtype=float); break
        if arr is None:
            for c in ms_cols:
                if c in df.columns:
                    arr = df[c].to_numpy(dtype=float) / 1000.0; break
        if arr is None:
            for c in df.columns:
                if pd.api.types.is_numeric_dtype(df[c]):
                    vals = df[c].to_numpy(dtype=float)
                    with np.errstate(all='ignore'):
                        med = np.nanmedian(vals)
                    if np.isfinite(med) and med > 200:  # likely ms -> convert to s
                        vals = vals / 1000.0
                    arr = vals; break
    if arr is None:
        raise RuntimeError("No usable numeric latency column found in CSV. "
                           f"Available columns: {list(df.columns)}")
    arr = arr[np.isfinite(arr)]
    arr = np.sort(arr)
    if warmup_frac > 0 and not keep_warmup:
        n = len(arr); cut = int(n * warmup_frac)
        if 0 < cut < n: arr = arr[cut:]
    return arr

def load_gev_results(path: str):
    # Expected columns per user:
    # timestamp,metric,scope,pod,n_blocks,shape_c,xi,location_mu,scale_beta,ks_D,ks_pvalue,config_hash
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    return df

lat_s = load_from_csv(BLOCK_MAXIMA_CSV, col_name=COL_NAME,
                      max_rows=MAX_ROWS, warmup_frac=WARMUP_FRAC, keep_warmup=KEEP_WARMUP,
                      sec_cols=SEC_COLS, ms_cols=MS_COLS)
gev_df = load_gev_results(GEV_RESULTS_CSV)

print("Loaded block maxima:", BLOCK_MAXIMA_CSV, "n=", len(lat_s))
print("Loaded GEV results:", "OK" if gev_df is not None else "not found")
if gev_df is not None:
    display(gev_df.head())


In [ ]:

# ==== EVT helpers ====

def exceedances(x: np.ndarray, u: float) -> np.ndarray:
    return x[x > u] - u

def gpd_fit(y: np.ndarray) -> Tuple[float, float]:
    c, loc, scale = genpareto.fit(y, floc=0.0)
    return float(c), float(scale)

def ks_test_gpd(y: np.ndarray, xi: float, sigma: float):
    cdf = lambda t: genpareto.cdf(t, c=xi, loc=0.0, scale=sigma)
    return kstest(y, cdf)

def anderson_darling_gof_uniform(uvals: np.ndarray) -> float:
    u = np.sort(np.clip(uvals, 1e-12, 1-1e-12)); n = len(u)
    if n == 0: return float("inf")
    i = np.arange(1, n+1)
    s = np.sum((2*i-1) * (np.log(u) + np.log(1 - u[::-1])))
    return float(-n - s / n)

def ad_test_gpd(y: np.ndarray, xi: float, sigma: float) -> float:
    U = genpareto.cdf(y, c=xi, loc=0.0, scale=sigma)
    return anderson_darling_gof_uniform(U)

def gpd_qq_plot(y: np.ndarray, xi: float, sigma: float, title: str, out_path: str):
    n = len(y)
    if n < 5: return
    y_sorted = np.sort(y)
    probs = (np.arange(1, n+1) - 0.5) / n
    q_theory = genpareto.ppf(probs, c=xi, loc=0.0, scale=sigma)
    plt.figure(figsize=(9, 6))
    plt.scatter(q_theory, y_sorted, s=18)
    lim = max(q_theory.max(), y_sorted.max())
    plt.plot([0, lim], [0, lim])
    plt.xlabel("Theoretical quantiles"); plt.ylabel("Empirical quantiles")
    plt.title(title); plt.tight_layout()
    plt.savefig(out_path, dpi=150); plt.close()


In [ ]:

# ==== Output folder ====
if OUT_DIR is None:
    OUT_DIR = os.path.join("/mnt/data", f"evt_from_blockmax_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(OUT_DIR, exist_ok=True)
print("Outputs will be saved to:", OUT_DIR)


In [ ]:

# ==== Mean Residual Life (MRL) ====
mrl_us = np.linspace(min(THRESHOLDS_S), max(THRESHOLDS_S), 25)
mrl_vals = []
for u in mrl_us:
    y = exceedances(lat_s, u)
    mrl_vals.append(np.mean(y) if len(y) else np.nan)

plt.figure(figsize=(9, 6))
plt.plot(mrl_us, mrl_vals, marker='o')
plt.xlabel("Threshold u (s)"); plt.ylabel("MRL E[X-u | X>u] (s)")
plt.title("Mean Residual Life (MRL) — from block_maxima.csv")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "mrl_curve.png"), dpi=150)
plt.show()


In [ ]:

# ==== Threshold-wise POT fits + QQ + GOF ====
rows, xi_us, xi_vals = [], [], []
for u in THRESHOLDS_S:
    y = exceedances(lat_s, u)
    n_exc = len(y)
    if n_exc < 5:
        rows.append(dict(u=u, n_exc=n_exc, xi=np.nan, sigma=np.nan, ks_stat=np.nan, ks_p=np.nan, ad_stat=np.nan))
        continue
    xi, sigma = gpd_fit(y)
    ks_stat, ks_p = ks_test_gpd(y, xi, sigma)
    ad_stat = ad_test_gpd(y, xi, sigma)

    xi_us.append(u); xi_vals.append(xi)

    gpd_qq_plot(y, xi, sigma, f"GPD QQ — u={u:.2f}s (n={n_exc})",
                os.path.join(OUT_DIR, f"qq_gpd_u_{str(u).replace('.','p')}.png"))

    rows.append(dict(u=u, n_exc=n_exc, xi=xi, sigma=sigma, ks_stat=ks_stat, ks_p=ks_p, ad_stat=ad_stat))

df_pot = pd.DataFrame(rows).sort_values("u")
display(df_pot)
df_pot.to_csv(os.path.join(OUT_DIR, "gpd_threshold_summary.csv"), index=False)


In [ ]:

# ==== ξ stability with optional GEV overlay ====

plt.figure(figsize=(9, 6))
if xi_us:
    plt.plot(xi_us, xi_vals, marker='o', label="POT ξ (GPD)")

overlay_loaded = False
if gev_df is not None and "xi" in gev_df.columns:
    # If 'u' exists, align by u; else draw a reference line by median
    if "u" in gev_df.columns:
        plt.plot(gev_df["u"].values, gev_df["xi"].values, marker='x', linestyle='--', label="GEV ξ (from CSV)")
        overlay_loaded = True
    else:
        xi_med = float(gev_df["xi"].median())
        plt.axhline(xi_med, linestyle='--', label=f"GEV ξ median ≈ {xi_med:.3f}")
        overlay_loaded = True

plt.xlabel("Threshold u (s)"); plt.ylabel("Shape ξ")
ttl = "Shape (ξ) stability — POT"
if overlay_loaded: ttl += " + GEV overlay"
plt.title(ttl); plt.legend(loc="best")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "xi_stability.png"), dpi=150)
plt.show()


In [ ]:

# ==== Save a small manifest ====
manifest = {
    "out_dir": OUT_DIR,
    "block_maxima_csv": BLOCK_MAXIMA_CSV,
    "gev_results_csv": GEV_RESULTS_CSV,
    "thresholds": THRESHOLDS_S,
    "generated": datetime.utcnow().isoformat() + "Z",
    "files": ["mrl_curve.png", "xi_stability.png", "gpd_threshold_summary.csv"]
          + sorted([f for f in os.listdir(OUT_DIR) if f.startswith("qq_gpd_u_")])
}
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("Done. Outputs in:", OUT_DIR)
